# H3: Joint Fitness Model Comparison

M4 (joint W with ω + κ) vs M1 (effort-only), M2 (threat-only), M3 (single-parameter).

**Note:** SVI results shown as exploratory benchmarks. Confirmatory uses MCMC WAIC + PSIS-LOO.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

REPO_ROOT = Path(os.getcwd())
for _ in range(5):
    if (REPO_ROOT / '.git').exists(): break
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

# Load MCMC results — combine latest run (M1, M3b, M5) with previous (M2, M3)
mc_new = pd.read_csv('results/stats/joint_optimal/mcmc_model_comparison.csv')

# M5 WAIC from latest run (hybrid model)
m5_waic = mc_new.loc[mc_new['Model']=='M5', 'WAIC'].values[0]
m5_acc = mc_new.loc[mc_new['Model']=='M5', 'choice_acc'].values[0]
m5_vig = mc_new.loc[mc_new['Model']=='M5', 'vigor_r2'].values[0]

# M1 from latest run
m1_waic = mc_new.loc[mc_new['Model']=='M1', 'WAIC'].values[0] if 'M1' in mc_new['Model'].values else np.nan
m1_acc = mc_new.loc[mc_new['Model']=='M1', 'choice_acc'].values[0] if 'M1' in mc_new['Model'].values else np.nan

# M2, M3 from previous run (pre-hybrid, but these models don't use total demand so values are comparable)
# These will be updated when M2/M3 are rerun on GPU with the hybrid M5
m2_waic = 14740.2  # from previous MCMC run
m2_acc = 0.789
m2_vig = 0.006
m3_waic = 15375.0  # from previous MCMC run
m3_acc = 0.773
m3_vig = 0.101

# M3b from latest run
m3b_waic = mc_new.loc[mc_new['Model']=='M3b', 'WAIC'].values[0] if 'M3b' in mc_new['Model'].values else np.nan

print('MCMC Model Comparison (Hybrid M5)')
print('='*60)
print(f'  M1 (effort-only):   WAIC={m1_waic:.0f}, acc={m1_acc:.3f}')
print(f'  M2 (threat-only):   WAIC={m2_waic:.0f}, acc={m2_acc:.3f}')
print(f'  M3 (single-param):  WAIC={m3_waic:.0f}, acc={m3_acc:.3f}')
print(f'  M5 (joint hybrid):  WAIC={m5_waic:.0f}, acc={m5_acc:.3f}, vig r²={m5_vig:.3f}')
if not np.isnan(m3b_waic):
    print(f'  M3b (scaled):       WAIC={m3b_waic:.0f} (supplementary)')


MCMC Model Comparison (Hybrid M5)
  M1 (effort-only):   WAIC=15060, acc=0.710
  M2 (threat-only):   WAIC=14740, acc=0.789
  M3 (single-param):  WAIC=15375, acc=0.773
  M5 (joint hybrid):  WAIC=12670, acc=0.780, vig r²=0.387
  M3b (scaled):       WAIC=14735 (supplementary)


## H3a: M4 vs M1 — Threat matters beyond effort

In [2]:
dw = m1_waic - m5_waic
passed = dw > 0
print('H3a — Joint vs Effort-only')
print('=' * 50)
print(f'  M1 WAIC: {m1_waic:.0f}, acc={m1_acc:.3f}')
print(f'  M5 WAIC: {m5_waic:.0f}, acc={m5_acc:.3f}')
print(f'  ΔWAIC = {dw:+.0f}')
print(f'  H3a: {"PASS ✓" if passed else "FAIL ✗"}')

H3a — Joint vs Effort-only
  M1 WAIC: 15060, acc=0.710
  M5 WAIC: 12670, acc=0.780
  ΔWAIC = +2390
  H3a: PASS ✓


## H3b: M4 vs M2 — Individual effort differences matter

In [3]:
dw = m2_waic - m5_waic
passed = dw > 0
print('H3b — Joint vs Threat-only')
print('=' * 50)
print(f'  M2 WAIC: {m2_waic:.0f}, acc={m2_acc:.3f}, vig r²={m2_vig:.3f}')
print(f'  M5 WAIC: {m5_waic:.0f}, acc={m5_acc:.3f}, vig r²={m5_vig:.3f}')
print(f'  ΔWAIC = {dw:+.0f}')
print(f'  H3b: {"PASS ✓" if passed else "FAIL ✗"}')

H3b — Joint vs Threat-only
  M2 WAIC: 14740, acc=0.789, vig r²=0.006
  M5 WAIC: 12670, acc=0.780, vig r²=0.387
  ΔWAIC = +2070
  H3b: PASS ✓


## H3c: M4 vs M3 — Capture cost and effort cost are separable

In [4]:
dw = m3_waic - m5_waic
passed = dw > 0
print('H3c — Joint vs Single-parameter')
print('=' * 50)
print(f'  M3 WAIC: {m3_waic:.0f}, acc={m3_acc:.3f}, vig r²={m3_vig:.3f}')
print(f'  M5 WAIC: {m5_waic:.0f}, acc={m5_acc:.3f}, vig r²={m5_vig:.3f}')
print(f'  ΔWAIC = {dw:+.0f}')
print(f'  H3c: {"PASS ✓" if passed else "FAIL ✗"}')
if not np.isnan(m3b_waic):
    dw_3b = m3b_waic - m5_waic
    print(f'\n  Supplementary: M3b (scaled) ΔWAIC = {dw_3b:+.0f} — scale mismatch is NOT the issue')

H3c — Joint vs Single-parameter
  M3 WAIC: 15375, acc=0.773, vig r²=0.101
  M5 WAIC: 12670, acc=0.780, vig r²=0.387
  ΔWAIC = +2705
  H3c: PASS ✓

  Supplementary: M3b (scaled) ΔWAIC = +2064 — scale mismatch is NOT the issue


## Summary

In [5]:
print('H3 SUMMARY (MCMC)')
print('=' * 60)
tests = [
    ('H3a', 'Effort-only', m1_waic - m5_waic),
    ('H3b', 'Threat-only', m2_waic - m5_waic),
    ('H3c', 'Single-param', m3_waic - m5_waic),
]
print(f'{"Test":<8} {"Alt":<15} {"ΔWAIC":>8} {"Pass":>6}')
print('-' * 40)
for h, alt, dw in tests:
    p = '✓' if dw > 0 else '✗'
    print(f'{h:<8} {alt:<15} {dw:>+8.0f} {p:>6}')
n_pass = sum(1 for *_, dw in tests if dw > 0)
print(f'\n{n_pass}/3 tests pass.')

H3 SUMMARY (MCMC)
Test     Alt                ΔWAIC   Pass
----------------------------------------
H3a      Effort-only        +2390      ✓
H3b      Threat-only        +2070      ✓
H3c      Single-param       +2705      ✓

3/3 tests pass.
